In [1]:
import pandas as pd
import gc

#### Прочитать *fines.csv*

In [2]:
df_fines = pd.read_csv('../data/fines.csv')

#### Итерации разными методами

- Цикл

In [3]:
df_fines_loop = df_fines.copy()

In [7]:
%%timeit

result = []
for i in range(len(df_fines_loop)):
    value = df_fines_loop.iloc[i]['Fines'] / df_fines_loop.iloc[i]['Refund'] * df_fines_loop.iloc[i]['Year']
    result.append(value)

df_fines_loop['calculated'] = result

745 ms ± 17.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


- Iterrows

In [8]:
df_fines_iter = df_fines.copy()

In [9]:
%%timeit

for ind, row in df_fines_iter.iterrows():
    df_fines_iter.loc[ind, 'Calculated'] = row['Fines'] / row['Refund'] * row['Year']

1.09 s ± 131 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


- Apply

In [10]:
df_fines_apply = df_fines.copy()

In [11]:
%%timeit

df_fines_apply['Calculated'] = df_fines_apply.apply(
    lambda row: row['Fines'] / row['Refund'] * row['Year'],
    axis=1
)

45.9 ms ± 4.99 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


- Series

In [12]:
df_fines_series = df_fines.copy()

In [13]:
%%timeit

df_fines_series['Calculated'] = df_fines_series['Fines'] / df_fines_series['Refund'] * df_fines_series['Year']

1.19 ms ± 19.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


- Values

In [14]:
df_fines_values = df_fines.copy()

In [15]:
%%timeit

df_fines_values['Calculated'] = (df_fines_values['Fines'].values / df_fines_values['Refund'] * df_fines_values['Year'])

2.09 ms ± 132 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


#### Индексация

In [17]:
%%timeit

df_fines[df_fines['CarNumber'] == 'O136HO197RUS']

2.8 ms ± 216 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


 - с установкой индекса

In [18]:
df_fines_index = df_fines.copy()
df_fines_index.set_index('CarNumber', inplace=True)

In [19]:
%%timeit

df_fines_index.loc['O136HO197RUS']

394 μs ± 61 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


#### Понижение разрядности типов

In [20]:
df_fines.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  930 non-null    str    
 1   Refund     930 non-null    int64  
 2   Fines      930 non-null    float64
 3   Make       930 non-null    str    
 4   Model      917 non-null    str    
 5   Year       930 non-null    int64  
dtypes: float64(1), int64(2), str(3)
memory usage: 174.8 KB


In [21]:
optimized_df = df_fines.copy()
for col in optimized_df.select_dtypes(include=['float64']).columns:
    optimized_df[col] = optimized_df[col].astype('float32')
for col in optimized_df.select_dtypes(include=['int64']).columns:
    optimized_df[col] = pd.to_numeric(optimized_df[col], downcast='integer')

In [22]:
optimized_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  930 non-null    str    
 1   Refund     930 non-null    int8   
 2   Fines      930 non-null    float32
 3   Make       930 non-null    str    
 4   Model      917 non-null    str    
 5   Year       930 non-null    int16  
dtypes: float32(1), int16(1), int8(1), str(3)
memory usage: 159.4 KB


#### Категории

In [23]:
optimized_df.dtypes

CarNumber        str
Refund          int8
Fines        float32
Make             str
Model            str
Year           int16
dtype: object

In [24]:
print(optimized_df.select_dtypes(include=['object']).columns.tolist())

['CarNumber', 'Make', 'Model']


/tmp/ipykernel_494557/4022712503.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(optimized_df.select_dtypes(include=['object']).columns.tolist())


In [25]:
for col in optimized_df.select_dtypes(include=['str']).columns:
    optimized_df[col] = optimized_df[col].astype('category')

In [26]:
optimized_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   CarNumber  930 non-null    category
 1   Refund     930 non-null    int8    
 2   Fines      930 non-null    float32 
 3   Make       930 non-null    category
 4   Model      917 non-null    category
 5   Year       930 non-null    int16   
dtypes: category(3), float32(1), int16(1), int8(1)
memory usage: 43.3 KB


> использование памяти уменьшилось со 159,4 KB до 43,3 KB

#### Очистка памяти

In [27]:
%reset_selective -f "^df_fines$"